In [ ]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import re
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pickle
import numpy as np
from collections import Counter, defaultdict
from konlpy.tag import Okt

# ========== (1) 날짜 파싱 ==========
def parse_naver_datetime(date_str):
    m = re.match(r"(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d+):(\d+)", str(date_str))
    if not m:
        return None
    year, month, day, ampm, hour, minute = m.groups()
    hour = int(hour)
    minute = int(minute)
    if ampm == '오후' and hour != 12:
        hour += 12
    if ampm == '오전' and hour == 12:
        hour = 0
    try:
        return datetime(int(year), int(month), int(day), hour, minute)
    except:
        return None

# ========== (2) 날짜 구간 계산 ==========
def get_custom_period(now=None):
    if now is None:
        now = datetime.now()
    end_dt = now.replace(hour=15, minute=30, second=0, microsecond=0) - timedelta(days=1)
    start_dt = now.replace(hour=15, minute=31, second=0, microsecond=0) - timedelta(days=4)
    return start_dt, end_dt

# ========== (3) 네이버 뉴스 크롤링 ==========
def crawl_multiple_cats_custom_clicks(cat_to_more_clicks):
    all_results = []
    start_dt, end_dt = get_custom_period()
    print(f"[INFO] 수집 구간: {start_dt} ~ {end_dt}")

    for cat, more_clicks in cat_to_more_clicks.items():
        print(f"\n[SECTION {cat}] 더보기 {more_clicks}번 클릭 시작")
        base_url = f'https://news.naver.com/section/{cat}'
        driver = webdriver.Chrome()
        driver.get(base_url)
        time.sleep(1)
        for i in range(more_clicks):
            try:
                more_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, '_CONTENT_LIST_LOAD_MORE_BUTTON'))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                time.sleep(0.2)
                ActionChains(driver).move_to_element(more_button).perform()
                time.sleep(0.1)
                more_button.click()
                print(f"[{cat}] 더보기 {i+1}번 클릭")
                time.sleep(1)
            except Exception as e:
                print(f'[{cat}] 더보기 버튼 클릭 실패:', e)
                break

        html = driver.page_source
        soup = BeautifulSoup(html, 'lxml')
        articles = soup.select('a.sa_text_title')
        print(f"[{cat}] 총 기사 수(목록): {len(articles)}")

        visited_urls = set()
        stop_flag = False
        for idx, article in enumerate(articles):
            article_url = article['href']
            if article_url in visited_urls:
                continue
            visited_urls.add(article_url)

            driver.get(article_url)
            time.sleep(0.5)
            article_soup = BeautifulSoup(driver.page_source, 'lxml')

            # 제목
            title_tag = article_soup.select_one('h2#title_area > span')
            title_text = title_tag.get_text(strip=True) if title_tag else '제목 없음'
            # 본문
            content_tag = article_soup.select_one('article#dic_area')
            content_text = content_tag.get_text(strip=True) if content_tag else ''
            # 날짜
            date_tag = article_soup.select_one('span._ARTICLE_DATE_TIME')
            date_text = date_tag.get_text(strip=True) if date_tag else ''
            article_dt = parse_naver_datetime(date_text)

            # ▶▶ 원하는 구간만 저장, 구간 이전 기사부터 중단
            if article_dt is not None:
                if start_dt <= article_dt <= end_dt:
                    all_results.append({
                        'cat': cat,
                        'date': date_text,
                        'title': title_text,
                        'content': content_text,
                        'url': article_url
                    })
                elif article_dt < start_dt:
                    print(f"[{cat}] {article_dt} → 구간 이전 기사! 섹션 크롤링 종료")
                    stop_flag = True
                    break

        driver.quit()
        print(f"[{cat}] 누적 기사 건수: {len(all_results)}")
        if stop_flag:
            continue

    return pd.DataFrame(all_results)

# ========== (4) 뉴스 전처리 ==========
def clean_naver_news_text(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r"[가-힣]{2,4}\s기자", " ", text)
    text = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", " ", text)
    text = re.sub(r"저작권자\s?.{0,20}(무단)?전재\s?(재)?배포(금지)?", " ", text)
    text = re.sub(r"(무단전재\s?및\s?재배포\s?금지)", " ", text)
    text = re.sub(r"제보는\s*카카오톡\s*okjebo.*", " ", text)
    text = re.sub(r"사진[=:/]?\s*[가-힣a-zA-Z0-9]*", " ", text)
    text = re.sub(r"\[사진\]", " ", text)
    text = re.sub(r"이미지\s*확대보기", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"\([^)]*기자[^)]*\)", " ", text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"▲.*", " ", text)
    text = re.sub(r"(관련기사|동영상|포토|SNS|카카오톡|페이스북|트위터|구독하기|영상취재|영상편집|디자인)\s?[가-힣]{2,4}", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ========== (5) BERT 산업군 분류 ==========
def predict_industry(text_list, model, tokenizer, label_encoder, device, batch_size=16):
    model.eval()
    pred_list = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=256
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            pred_list.extend(label_encoder.inverse_transform(preds))
    return pred_list

# ========== (6) 감정 사전/산업별 단어 로딩 ==========
def load_sentiment_resources(lexicon_path, industry_word_path):
    lexicon_df = pd.read_csv(lexicon_path, encoding='utf-8')
    lexicon_df['score'] = pd.to_numeric(lexicon_df['score'], errors='coerce')
    lexicon_df.dropna(subset=['score'], inplace=True)
    lexicon = dict(zip(lexicon_df['word'], lexicon_df['score']))

    industry_word_df = pd.read_excel(industry_word_path)
    industry_word_dict = industry_word_df.groupby('industry')['word'].apply(set).to_dict()

    return lexicon, industry_word_dict

# ========== (7) 감정 분석 ==========
def compute_sentiment_score(text, lexicon, industry=None, industry_word_dict=None, industry_weight=1.5):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, ''

    NEGATION_WORDS = ['않다', '못하다', '없다', '아니다', '줄이다']
    NEGATIVE_CONTEXT = ['문제', '위기', '불안', '침체', '파산', '위협', '적자', '부진', '하락', '리스크']
    POSITIVE_RESOLVE = ['해결', '극복', '해소', '회복', '정상화', '반등', '개선']
    NEGATE_IGNORE_WORDS = ['전쟁', '위기', '불확실', '파산', '침체', '리스크']

    okt = Okt()
    text_len = len(text)
    if text_len <= 625:
        per_word_cap = 1.5
    elif text_len <= 1360:
        per_word_cap = 2.5
    else:
        per_word_cap = 3.5

    word_score_dict = defaultdict(float)
    word_count_dict = defaultdict(int)
    industry_words = set()
    if industry_word_dict and industry:
        industry_words = industry_word_dict.get(industry, set())

    # 복합어 처리
    for phrase, score in lexicon.items():
        if ' ' in phrase and phrase in text:
            base_score = score
            if phrase in industry_words:
                base_score *= industry_weight
            capped = max(min(base_score, per_word_cap), -per_word_cap)
            word_score_dict[phrase] += capped
            word_score_dict[phrase] = max(min(word_score_dict[phrase], per_word_cap), -per_word_cap)
            word_count_dict[phrase] += 1
            text = text.replace(phrase, '')

    def split_sentences(text):
        return re.split(r'(?<=[.!?])\s+', str(text))

    # 단일어 분석
    for sentence in split_sentences(text):
        try:
            tokens = okt.pos(sentence, stem=True)
        except:
            continue
        words = [w for w, _ in tokens]
        word_freq = Counter(words)
        for i, (word, pos) in enumerate(tokens):
            if pos not in ['Noun', 'Adjective', 'Verb']:
                continue
            if word not in lexicon:
                continue
            base_score = lexicon[word]
            if word in industry_words:
                base_score *= industry_weight
            count = word_freq[word]
            if word in NEGATE_IGNORE_WORDS:
                adjusted_score = base_score
            else:
                if pos in ['Verb', 'Adjective']:
                    negate = any(
                        (i + j < len(tokens) and tokens[i + j][0] in NEGATION_WORDS) or
                        (i - j >= 0 and tokens[i - j][0] in NEGATION_WORDS)
                        for j in range(1, 2)
                    )
                else:
                    negate = False
                adjusted_score = base_score * (-1 if negate else 1)
            log_score = adjusted_score * np.log10(1 + count) * 0.8
            capped_log_score = max(min(log_score, per_word_cap), -per_word_cap)
            word_score_dict[word] += capped_log_score
            word_score_dict[word] = max(min(word_score_dict[word], per_word_cap), -per_word_cap)
            word_count_dict[word] += count

        # 긍정 해결 단어로 보정
        if any(w in words for w in NEGATIVE_CONTEXT) and any(p in words for p in POSITIVE_RESOLVE):
            word_score_dict['보정'] += 1.5
            word_score_dict['보정'] = min(word_score_dict['보정'], per_word_cap)
            word_count_dict['보정'] += 1

    total_score = sum(word_score_dict.values())
    hit_summary = sorted(
        [f"{word}:{round(word_score_dict[word], 3)}({word_count_dict[word]})" for word in word_score_dict],
        key=lambda x: abs(float(x.split(":")[1].split("(")[0])),
        reverse=True
    )

    return round(total_score, 3), ', '.join(hit_summary)

# ========== (8) 감정 점수 DataFrame 추가 ==========
def add_sentiment_scores(df, lexicon, industry_word_dict):
    results = df.apply(
        lambda row: compute_sentiment_score(
            row['content'],
            lexicon,
            row.get('industry'),
            industry_word_dict=industry_word_dict
        ),
        axis=1,
        result_type='expand'
    )
    df[['sentiment_score', 'hit_words']] = results
    df['content_length'] = df['content'].str.len().replace(0, np.nan)
    df['normalized_score'] = df['sentiment_score'] / np.log10(df['content_length'] + 1)
    return df

# ========== (9) 전체 파이프라인 ==========
def run_full_pipeline(cat_to_more_clicks, outname=None,
                      model=None, tokenizer=None, label_encoder=None, device=None,
                      lexicon=None, industry_word_dict=None):
    # 1. 크롤링
    df = crawl_multiple_cats_custom_clicks(cat_to_more_clicks)
    # 2. 전처리
    df['clean_content'] = df['content'].apply(clean_naver_news_text)
    # 3. BERT (산업군 분류)
    df['industry'] = predict_industry(
        df['clean_content'].tolist(),
        model, tokenizer, label_encoder, device
    )
    # 4. 감정분석
    df = add_sentiment_scores(df, lexicon, industry_word_dict)
    # 5. 저장
    today_str = datetime.now().strftime('%y%m%d')
    if not outname:
        outname = f'naver_news_{today_str}_3day.csv'
    df.to_csv(outname, index=False, encoding='utf-8-sig')
    print(f"[DONE] 저장: {outname} (총 {len(df)}건)")
    return df

# ========== (10) 리소스 로딩 및 실행 ==========
# checkpoint_dir = r"./kobert_output/checkpoint-3505"
checkpoint_dir = r"C:\Users\KI\Documents\data\ML\project\realtime\kobert_output\checkpoint-3505"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
with open(os.path.join(checkpoint_dir, "label_encoder.pkl"), "rb") as f:
    label_encoder = pickle.load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

lexicon, industry_word_dict = load_sentiment_resources(
    os.path.join(checkpoint_dir, "finance_sentiment_cleaned_final_700.csv"),
    os.path.join(checkpoint_dir, "word_of_industry_v3.xlsx")
)

cat_to_more_clicks = {
    '101': 99 # 테스트용
    '105': 50,
    '104': 50
}
df_result = run_full_pipeline(
    cat_to_more_clicks,
    model=model, tokenizer=tokenizer, label_encoder=label_encoder, device=device,
    lexicon=lexicon, industry_word_dict=industry_word_dict
)


In [3]:
import pandas as pd
import time
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import re
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pickle
import numpy as np
from collections import Counter, defaultdict
from konlpy.tag import Okt

# ========== (1) 날짜 파싱 ==========
def parse_naver_datetime(date_str):
    m = re.match(r"(\d{4})\.(\d{2})\.(\d{2})\.\s*(오전|오후)\s*(\d+):(\d+)", str(date_str))
    if not m:
        return None
    year, month, day, ampm, hour, minute = m.groups()
    hour = int(hour)
    minute = int(minute)
    if ampm == '오후' and hour != 12:
        hour += 12
    if ampm == '오전' and hour == 12:
        hour = 0
    try:
        return datetime(int(year), int(month), int(day), hour, minute)
    except:
        return None

# ========== (2) 날짜 구간 계산 ==========
def get_custom_period(now=None):
    if now is None:
        now = datetime.now()
    end_dt = now.replace(hour=15, minute=30, second=0, microsecond=0) - timedelta(days=1)
    start_dt = now.replace(hour=15, minute=31, second=0, microsecond=0) - timedelta(days=4)
    return start_dt, end_dt

# ========== (3) 네이버 뉴스 크롤링 ==========
def crawl_multiple_cats_custom_clicks(cat_to_more_clicks):
    all_results = []
    start_dt, end_dt = get_custom_period()
    print(f"[INFO] 수집 구간: {start_dt} ~ {end_dt}")

    for cat, more_clicks in cat_to_more_clicks.items():
        print(f"\n[SECTION {cat}] 더보기 {more_clicks}번 클릭 시작")
        base_url = f'https://news.naver.com/section/{cat}'
        driver = webdriver.Chrome()
        driver.get(base_url)
        time.sleep(1)
        for i in range(more_clicks):
            try:
                more_button = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, '_CONTENT_LIST_LOAD_MORE_BUTTON'))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                time.sleep(0.2)
                ActionChains(driver).move_to_element(more_button).perform()
                time.sleep(0.1)
                more_button.click()
                print(f"[{cat}] 더보기 {i+1}번 클릭")
                time.sleep(1)
            except Exception as e:
                print(f'[{cat}] 더보기 버튼 클릭 실패:', e)
                break

        html = driver.page_source
        soup = BeautifulSoup(html, 'lxml')
        articles = soup.select('a.sa_text_title')
        print(f"[{cat}] 총 기사 수(목록): {len(articles)}")

        visited_urls = set()
        stop_flag = False
        for idx, article in enumerate(articles):
            article_url = article['href']
            if article_url in visited_urls:
                continue
            visited_urls.add(article_url)

            driver.get(article_url)
            time.sleep(0.5)
            article_soup = BeautifulSoup(driver.page_source, 'lxml')

            # 제목
            title_tag = article_soup.select_one('h2#title_area > span')
            title_text = title_tag.get_text(strip=True) if title_tag else '제목 없음'
            # 본문
            content_tag = article_soup.select_one('article#dic_area')
            content_text = content_tag.get_text(strip=True) if content_tag else ''
            # 날짜
            date_tag = article_soup.select_one('span._ARTICLE_DATE_TIME')
            date_text = date_tag.get_text(strip=True) if date_tag else ''
            article_dt = parse_naver_datetime(date_text)

            # ▶▶ 원하는 구간만 저장, 구간 이전 기사부터 중단
            if article_dt is not None:
                if start_dt <= article_dt <= end_dt:
                    all_results.append({
                        'cat': cat,
                        'date_raw': date_text,      # 원본 날짜 텍스트 저장(변환 전)
                        'parsed_dt': article_dt,    # datetime으로 저장
                        'title': title_text,
                        'content': content_text,
                        'url': article_url
                    })
                elif article_dt < start_dt:
                    print(f"[{cat}] {article_dt} → 구간 이전 기사! 섹션 크롤링 종료")
                    stop_flag = True
                    break

        driver.quit()
        print(f"[{cat}] 누적 기사 건수: {len(all_results)}")
        if stop_flag:
            continue

    return pd.DataFrame(all_results)

# ========== (4) 날짜 변환 함수 ==========
def get_news_base_date(article_dt, now=None):
    """
    기사 datetime이 '어제 15:31' 이후면 오늘 날짜,
    아니면 어제 날짜를 반환
    """
    if article_dt is None:
        return ""
    if now is None:
        now = datetime.now()
    cutoff = now.replace(hour=15, minute=31, second=0, microsecond=0) - timedelta(days=1)
    if article_dt > cutoff:
        return now.strftime("%Y-%m-%d")  # 오늘 날짜
    else:
        return (now - timedelta(days=1)).strftime("%Y-%m-%d")  # 어제 날짜

def get_adjusted_date(article_dt):
    if article_dt is None:
        return ""
    return article_dt.strftime("%d/%m/%Y %H:%M")

# ========== (5) 뉴스 전처리 ==========
def clean_naver_news_text(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r"[가-힣]{2,4}\s기자", " ", text)
    text = re.sub(r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+", " ", text)
    text = re.sub(r"저작권자\s?.{0,20}(무단)?전재\s?(재)?배포(금지)?", " ", text)
    text = re.sub(r"(무단전재\s?및\s?재배포\s?금지)", " ", text)
    text = re.sub(r"제보는\s*카카오톡\s*okjebo.*", " ", text)
    text = re.sub(r"사진[=:/]?\s*[가-힣a-zA-Z0-9]*", " ", text)
    text = re.sub(r"\[사진\]", " ", text)
    text = re.sub(r"이미지\s*확대보기", " ", text)
    text = re.sub(r"\[[^\]]*\]", " ", text)
    text = re.sub(r"\([^)]*기자[^)]*\)", " ", text)
    text = re.sub(r"\([^)]*\)", " ", text)
    text = re.sub(r"▲.*", " ", text)
    text = re.sub(r"(관련기사|동영상|포토|SNS|카카오톡|페이스북|트위터|구독하기|영상취재|영상편집|디자인)\s?[가-힣]{2,4}", " ", text)
    text = re.sub(r"[^가-힣a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ========== (6) BERT 산업군 분류 ==========
def predict_industry(text_list, model, tokenizer, label_encoder, device, batch_size=16):
    model.eval()
    pred_list = []
    with torch.no_grad():
        for i in range(0, len(text_list), batch_size):
            batch = text_list[i:i+batch_size]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=256
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            pred_list.extend(label_encoder.inverse_transform(preds))
    return pred_list

# ========== (7) 감정 사전/산업별 단어 로딩 ==========
def load_sentiment_resources(lexicon_path, industry_word_path):
    lexicon_df = pd.read_csv(lexicon_path, encoding='utf-8')
    lexicon_df['score'] = pd.to_numeric(lexicon_df['score'], errors='coerce')
    lexicon_df.dropna(subset=['score'], inplace=True)
    lexicon = dict(zip(lexicon_df['word'], lexicon_df['score']))

    industry_word_df = pd.read_excel(industry_word_path)
    industry_word_dict = industry_word_df.groupby('industry')['word'].apply(set).to_dict()

    return lexicon, industry_word_dict

# ========== (8) 감정 분석 ==========
def compute_sentiment_score(text, lexicon, industry=None, industry_word_dict=None, industry_weight=1.5):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return 0.0, ''

    NEGATION_WORDS = ['않다', '못하다', '없다', '아니다', '줄이다']
    NEGATIVE_CONTEXT = ['문제', '위기', '불안', '침체', '파산', '위협', '적자', '부진', '하락', '리스크']
    POSITIVE_RESOLVE = ['해결', '극복', '해소', '회복', '정상화', '반등', '개선']
    NEGATE_IGNORE_WORDS = ['전쟁', '위기', '불확실', '파산', '침체', '리스크']

    okt = Okt()
    text_len = len(text)
    if text_len <= 625:
        per_word_cap = 1.5
    elif text_len <= 1360:
        per_word_cap = 2.5
    else:
        per_word_cap = 3.5

    word_score_dict = defaultdict(float)
    word_count_dict = defaultdict(int)
    industry_words = set()
    if industry_word_dict and industry:
        industry_words = industry_word_dict.get(industry, set())

    # 복합어 처리
    for phrase, score in lexicon.items():
        if ' ' in phrase and phrase in text:
            base_score = score
            if phrase in industry_words:
                base_score *= industry_weight
            capped = max(min(base_score, per_word_cap), -per_word_cap)
            word_score_dict[phrase] += capped
            word_score_dict[phrase] = max(min(word_score_dict[phrase], per_word_cap), -per_word_cap)
            word_count_dict[phrase] += 1
            text = text.replace(phrase, '')

    def split_sentences(text):
        return re.split(r'(?<=[.!?])\s+', str(text))

    # 단일어 분석
    for sentence in split_sentences(text):
        try:
            tokens = okt.pos(sentence, stem=True)
        except:
            continue
        words = [w for w, _ in tokens]
        word_freq = Counter(words)
        for i, (word, pos) in enumerate(tokens):
            if pos not in ['Noun', 'Adjective', 'Verb']:
                continue
            if word not in lexicon:
                continue
            base_score = lexicon[word]
            if word in industry_words:
                base_score *= industry_weight
            count = word_freq[word]
            if word in NEGATE_IGNORE_WORDS:
                adjusted_score = base_score
            else:
                if pos in ['Verb', 'Adjective']:
                    negate = any(
                        (i + j < len(tokens) and tokens[i + j][0] in NEGATION_WORDS) or
                        (i - j >= 0 and tokens[i - j][0] in NEGATION_WORDS)
                        for j in range(1, 2)
                    )
                else:
                    negate = False
                adjusted_score = base_score * (-1 if negate else 1)
            log_score = adjusted_score * np.log10(1 + count) * 0.8
            capped_log_score = max(min(log_score, per_word_cap), -per_word_cap)
            word_score_dict[word] += capped_log_score
            word_score_dict[word] = max(min(word_score_dict[word], per_word_cap), -per_word_cap)
            word_count_dict[word] += count

        # 긍정 해결 단어로 보정
        if any(w in words for w in NEGATIVE_CONTEXT) and any(p in words for p in POSITIVE_RESOLVE):
            word_score_dict['보정'] += 1.5
            word_score_dict['보정'] = min(word_score_dict['보정'], per_word_cap)
            word_count_dict['보정'] += 1

    total_score = sum(word_score_dict.values())
    hit_summary = sorted(
        [f"{word}:{round(word_score_dict[word], 3)}({word_count_dict[word]})" for word in word_score_dict],
        key=lambda x: abs(float(x.split(":")[1].split("(")[0])),
        reverse=True
    )

    return round(total_score, 3), ', '.join(hit_summary)

# ========== (9) 감정 점수 DataFrame 추가 ==========
def add_sentiment_scores(df, lexicon, industry_word_dict):
    results = df.apply(
        lambda row: compute_sentiment_score(
            row['content'],
            lexicon,
            row.get('industry'),
            industry_word_dict=industry_word_dict
        ),
        axis=1,
        result_type='expand'
    )
    df[['sentiment_score', 'hit_words']] = results
    df['content_length'] = df['content'].str.len().replace(0, np.nan)
    df['normalized_score'] = df['sentiment_score'] / np.log10(df['content_length'] + 1)
    return df

# ========== (10) 전체 파이프라인 ==========
def run_full_pipeline(cat_to_more_clicks, outname=None,
                      model=None, tokenizer=None, label_encoder=None, device=None,
                      lexicon=None, industry_word_dict=None):
    # 1. 크롤링
    df = crawl_multiple_cats_custom_clicks(cat_to_more_clicks)
    # 2. 날짜 컬럼 가공(병합용, 사람이 읽기좋은 포맷)
    df['adjusted_date'] = df['parsed_dt'].apply(get_adjusted_date)
    df['date'] = df['parsed_dt'].apply(get_news_base_date)
    # 3. 전처리
    df['clean_content'] = df['content'].apply(clean_naver_news_text)
    # 4. BERT (산업군 분류)
    df['industry'] = predict_industry(
        df['clean_content'].tolist(),
        model, tokenizer, label_encoder, device
    )
    # 5. 감정분석
    df = add_sentiment_scores(df, lexicon, industry_word_dict)
    # 6. 저장
    today_str = datetime.now().strftime('%y%m%d')
    if not outname:
        outname = f'naver_news_{today_str}_with_3day.csv'
    df.to_csv(outname, index=False, encoding='utf-8-sig')
    print(f"[DONE] 저장: {outname} (총 {len(df)}건)")
    return df

# ========== (11) 리소스 로딩 및 실행 ==========
checkpoint_dir = r"C:\Users\KI\Documents\data\ML\project\realtime\kobert_output\checkpoint-3505"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir)
with open(os.path.join(checkpoint_dir, "label_encoder.pkl"), "rb") as f:
    label_encoder = pickle.load(f)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

lexicon, industry_word_dict = load_sentiment_resources(
    os.path.join(checkpoint_dir, "finance_sentiment_cleaned_final_700.csv"),
    os.path.join(checkpoint_dir, "word_of_industry_v3.xlsx")
)

cat_to_more_clicks = {
    '101': 200,  # 테스트 및 충분한 기사 확보용
    '105': 100,
    '104': 100
}
df_result = run_full_pipeline(
    cat_to_more_clicks,
    model=model, tokenizer=tokenizer, label_encoder=label_encoder, device=device,
    lexicon=lexicon, industry_word_dict=industry_word_dict
)


[INFO] 수집 구간: 2025-07-11 15:31:00 ~ 2025-07-14 15:30:00

[SECTION 101] 더보기 200번 클릭 시작
[101] 더보기 1번 클릭
[101] 더보기 2번 클릭
[101] 더보기 3번 클릭
[101] 더보기 4번 클릭
[101] 더보기 5번 클릭
[101] 더보기 6번 클릭
[101] 더보기 7번 클릭
[101] 더보기 8번 클릭
[101] 더보기 9번 클릭
[101] 더보기 10번 클릭
[101] 더보기 11번 클릭
[101] 더보기 12번 클릭
[101] 더보기 13번 클릭
[101] 더보기 14번 클릭
[101] 더보기 15번 클릭
[101] 더보기 16번 클릭
[101] 더보기 17번 클릭
[101] 더보기 18번 클릭
[101] 더보기 19번 클릭
[101] 더보기 20번 클릭
[101] 더보기 21번 클릭
[101] 더보기 22번 클릭
[101] 더보기 23번 클릭
[101] 더보기 24번 클릭
[101] 더보기 25번 클릭
[101] 더보기 26번 클릭
[101] 더보기 27번 클릭
[101] 더보기 28번 클릭
[101] 더보기 29번 클릭
[101] 더보기 30번 클릭
[101] 더보기 31번 클릭
[101] 더보기 32번 클릭
[101] 더보기 33번 클릭
[101] 더보기 34번 클릭
[101] 더보기 35번 클릭
[101] 더보기 36번 클릭
[101] 더보기 37번 클릭
[101] 더보기 38번 클릭
[101] 더보기 39번 클릭
[101] 더보기 40번 클릭
[101] 더보기 41번 클릭
[101] 더보기 42번 클릭
[101] 더보기 43번 클릭
[101] 더보기 44번 클릭
[101] 더보기 45번 클릭
[101] 더보기 46번 클릭
[101] 더보기 47번 클릭
[101] 더보기 48번 클릭
[101] 더보기 49번 클릭
[101] 더보기 50번 클릭
[101] 더보기 51번 클릭
[101] 더보기 52번 클릭
[101] 더보기 53번 클릭
[101] 더보기 54번 클릭
[101]